# Example 1.2.1 / 5: check Statements

Companion notebook for **1.2.1 Restrictions, Defaults and Checks** in the AMPL lab Documentation, section "`check` Statements". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## `check` Statements

amplpy has no method for the `check` command, so `run.py` sends it with `ampl.eval("check;")`.
A failed check is reported as a warning: AMPL prints its message, no exception is raised, and
the program goes on, here to `ampl.get_value()`, which evaluates one expression:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set ORIG;
set DEST;
param supply {ORIG} >= 0;
param demand {DEST} >= 0;
check: sum {i in ORIG} supply[i] = sum {j in DEST} demand[j];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["ORIG"] = ["GARY", "PITT"]
ampl.set["DEST"] = ["FRA", "DET"]
ampl.param["supply"] = {"GARY": 1400, "PITT": 2900}
ampl.param["demand"] = {"FRA": 900, "DET": 1200}

In [5]:
# run.py
ampl.eval("check;")
print(ampl.get_value("sum {i in ORIG} supply[i]"))

check at line 5 of model.mod fails:
  check: sum{i in ORIG} supply[i] == sum{j in DEST} demand[j];
4300


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`. In a script run by the AMPL command line,
the failed `check` also stops everything after it:

In [6]:
%%writefile data.dat
set ORIG := GARY PITT;
set DEST := FRA DET;
param supply := GARY 1400  PITT 2900;
param demand := FRA 900    DET 1200;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
check;

check at line 5 of model.mod fails:
  check: sum{i in ORIG} supply[i] == sum{j in DEST} demand[j];
